<img src="img/Logo_UC3M.svg.png" width="150">

### Aprendizaje Automático · Grado en Ingeniería Informática · Curso 2025/26
# Práctica 2: Determinación de tipos de estrellas
Wenjie Chen y Antonio Grial Vázquez González

## 1. Codificación de datos
Lo primero que haces es cargar los datos y transformar las variables categóricas através de codificación ordinal ya que el color y la clase espectral tienen una jerarquía física.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

# 1. Cargar el dataset
df = pd.read_csv("starts_data.csv")

# 2. Limpieza de la variable 'Color'
# Unificamos los datos para que salgan de una misma forma
df['Color'] = df['Color'].str.lower().str.replace('-', ' ').str.strip()

# Mapeo de unificación para agrupar términos sinónimos o muy similares
color_unification = {
    'blue white': 'blue-white',
    'blue-white': 'blue-white',
    'pale yellow orange': 'yellow-orange',
    'white-yellow': 'yellow-white',
    'yellowish white': 'yellow-white',
    'yellowish': 'yellow',
    'whitish': 'white'
}
df['Color'] = df['Color'].replace(color_unification)

# 3. Definición del orden jerárquico (Ordinal)
# Clase Espectral: De la más caliente (O) a la más fría (M)
spectral_order = ['O', 'B', 'A', 'F', 'G', 'K', 'M']

# Color: De mayor energía/temperatura (Blue) a menor (Red)
color_order = [
    'blue', 
    'blue-white', 
    'white', 
    'yellow-white', 
    'yellow', 
    'yellow-orange', 
    'orange', 
    'red'
]

# 4. Aplicar OrdinalEncoder
# Creamos el codificador con los órdenes definidos
encoder = OrdinalEncoder(categories=[spectral_order, color_order])

# Aplicamos la transformación
df[['Spectral_Class', 'Color']] = encoder.fit_transform(df[['Spectral_Class', 'Color']])

# 5. Verificación rápida
print("Primeras filas con codificación ordinal:")
print(df[['Color', 'Spectral_Class']].head())

# Guardamos el NIA para la semilla de aleatoriedad
NIA_SEED = 100522255

## 2. Reducción de Dimensionalidad (PCA)
El PCA es una técnica de reducción de dimensionalidad que transforma las variables originales en un nuevo conjunto de variables llamadas Componentes Principales (PCs). Para este caso reducimos de 6 dimensiones a solo 2 dimensiones.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Selección de variables para el modelo
# Usamos todas las variables que ya hemos preprocesado (incluyendo las numéricas originales)
features = ['Temperature', 'L', 'R', 'A_M', 'Color', 'Spectral_Class']
X = df[features]

# 2. Estandarización (Media = 0, Varianza = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Aplicación de PCA para reducir a 2 dimensiones
# Fijamos random_state con el NIA del grupo
pca = PCA(n_components=2, random_state=NIA_SEED)
X_pca = pca.fit_transform(X_scaled)

# 4. Crear un DataFrame con los resultados para facilitar el manejo posterior
df_pca = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'])

# 5. Visualización de los datos transformados
plt.figure(figsize=(8, 6))
plt.scatter(df_pca['PC1'], df_pca['PC2'], c='blue', alpha=0.5, edgecolors='k')
plt.title('Proyección de los datos en las 2 Componentes Principales (PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True)
plt.show()

## 3. Aplicación de algoritmos de clustering
En esta sección aplicaremos los algoritmos directamente sobre las dos componentes principales obtenidas en el paso anterior.
### K-Means
Es un algoritmo basado en centroides. Usaremos el Coeficiente de Silueta para determinar el número óptimo de clusters (k),

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Probamos diferentes valores de K
range_n_clusters = range(2, 11)
best_k = 0
best_score = -1

for n_clusters in range_n_clusters:
    clusterer = KMeans(n_clusters=n_clusters, random_state=NIA_SEED, n_init=10)
    cluster_labels = clusterer.fit_predict(df_pca)
    
    # Calculamos la métrica de Silueta
    score = silhouette_score(df_pca, cluster_labels)
    print(f"Para n_clusters = {n_clusters}, el Silhouette Score es: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = n_clusters

print(f"\nEl número óptimo de clusters para K-Means es: {best_k}")

### Hierarchical Clustering/Dendrogramas
Este método permite ver la estructura de "árbol" de los datos. Usaremos el método de Ward porque minimiza la varianza dentro de los clusters, lo cual suele dar grupos más equilibrados.

In [ ]:
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering

# Visualización del Dendrograma para decidir el corte
plt.figure(figsize=(10, 7))
dendrogram = sch.dendrogram(sch.linkage(df_pca, method='ward'))
plt.title('Dendrograma de las Estrellas')
plt.xlabel('Índice de la Estrella')
plt.ylabel('Distancia Euclidiana')
plt.show()

# Aplicamos el modelo
hc = AgglomerativeClustering(n_clusters=best_k, metric='euclidean', linkage='ward')
hc_labels = hc.fit_predict(df_pca)

### DBSCAN
Se busca zonas de alta densidad. Para este algoritmo, usamos la métrica DBCV.

In [ ]:
from sklearn.cluster import DBSCAN
import hdbscan

# Probamos valores pequeños para eps y min_samples
eps_values = [0.3, 0.5, 0.7]
min_samples_values = [3, 5, 10]

for eps in eps_values:
    for min_s in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_s)
        labels = dbscan.fit_predict(df_pca)
        
        # Solo calculamos si hay más de 1 cluster y no todo es ruido (-1)
        if len(set(labels)) > 1:
            score = hdbscan.validity.dbcv(df_pca.values, labels)
            print(f"EPS: {eps}, MinSamples: {min_s} -> DBCV: {score:.4f}")